# Upload Finetuned T3 Model from checkpoint to HuggingFace

This notebook uploads a complete TTS model to HuggingFace with a finetuned T3 audio token generator from a checkpoint.

## 1. Configuration

Set your paths and HuggingFace repository details.

In [ ]:
import os
from huggingface_hub import snapshot_download
from huggingface_hub.constants import HF_HUB_CACHE
from chatterbox.tts_turbo import REPO_ID as TURBO_REPO_ID
from chatterbox.mtl_tts import REPO_ID as MTL_REPO_ID

IS_TURBO = False

# Path to the checkpoint containing the finetuned T3 model.safetensors
checkpoint_path = "checkpoint"  # UPDATE THIS
# HuggingFace repo name (will be created if it doesn't exist)
hf_repo_name = "upload/path"  # UPDATE THIS
# Whether to make the repo private
private_repo = True


if IS_TURBO:
    # Path to the base/complete model you want to upload (with all files)
    base_model_path = snapshot_download(
        repo_id=TURBO_REPO_ID,
        token=os.getenv("HF_TOKEN") or True,
        # Optional: Filter to download only what you need
        allow_patterns=["*.safetensors", "*.json", "*.txt", "*.pt", "*.model"],
        cache_dir=HF_HUB_CACHE,
    )
else: 
    base_model_path = snapshot_download(
        repo_id=MTL_REPO_ID,
        repo_type="model",
        revision="main", 
        allow_patterns=["ve.pt", "t3_mtl23ls_v2.safetensors", "s3gen.pt", "grapheme_mtl_merged_expanded_v1.json", "conds.pt", "Cangjie5_TC.json"],
        token=os.getenv("HF_TOKEN"),
    )

## 2. Import Libraries

In [ ]:
import os
import shutil
from pathlib import Path
from huggingface_hub import HfApi, create_repo, upload_folder
import tempfile
from safetensors.torch import load_file, save_file
import torch

## 3. Verify Paths

In [ ]:
# Verify checkpoint exists
checkpoint_model_path = Path(checkpoint_path) / "model.safetensors"
assert checkpoint_model_path.exists(), f"Checkpoint model not found at {checkpoint_model_path}"

# Verify base model exists
assert Path(base_model_path).exists(), f"Base model path not found at {base_model_path}"

print(f"✓ Checkpoint path: {checkpoint_path}")
print(f"✓ Checkpoint model: {checkpoint_model_path}")
print(f"✓ Base model path: {base_model_path}")

## 4. Create Temporary Upload Directory

Copy the base model and replace T3 with the finetuned version.

In [ ]:
  # Create temporary directory for the upload
temp_dir = tempfile.mkdtemp(prefix="hf_upload_")
print(f"Created temporary directory: {temp_dir}")

# Copy only top-level files from base model (skip subdirectories)
print("Copying base model files (top-level only)...")
for src_path in Path(base_model_path).iterdir():
    if src_path.is_file():
        shutil.copy2(src_path, Path(temp_dir) / src_path.name)
print(f"✓ Copied top-level files from {base_model_path} to {temp_dir}")

# Find the original T3 model filename in the base model directory
original_t3_filename = None
for src_file in Path(base_model_path).iterdir():
    if src_file.is_file() and src_file.name.startswith("t3") and src_file.suffix == ".safetensors":
        original_t3_filename = src_file.name
        print(f"Found original T3 model file: {original_t3_filename}")
        break

if not original_t3_filename:
    raise FileNotFoundError("Could not find T3 model file (t3*.safetensors) in base model directory")

# Load the checkpoint
print(f"Loading checkpoint from {checkpoint_model_path}...")
checkpoint_state_dict = load_file(checkpoint_model_path)

# Extract T3 weights (remove 't3.' prefix)
print("Extracting T3 weights from checkpoint...")
t3_state_dict = {}
for key, value in checkpoint_state_dict.items():
    if key.startswith('t3.'):
        t3_state_dict[key[3:]] = value

if not t3_state_dict:
    print("No 't3.' prefix found, assuming all weights are T3 weights...")
    t3_state_dict = checkpoint_state_dict

print(f"Extracted {len(t3_state_dict)} T3 weights")

if IS_TURBO:

    # Add back tfmr.wte.weight which was deleted during model loading but is required for loading
    # This embedding is not actually used (T3 uses custom text_emb/speech_emb instead), but
    # the loading code expects it to exist before deleting it
    if "tfmr.wte.weight" not in t3_state_dict:
        # Load the original T3 model to get vocab_size and hidden_size
        original_t3_path = Path(base_model_path) / original_t3_filename
        original_t3_state_dict = load_file(original_t3_path)
        
        # Get dimensions from text_emb or speech_emb layer
        if "text_emb.weight" in original_t3_state_dict:
            vocab_size, hidden_size = original_t3_state_dict["text_emb.weight"].shape
        elif "tfmr.wte.weight" in original_t3_state_dict:
            vocab_size, hidden_size = original_t3_state_dict["tfmr.wte.weight"].shape
        else:
            # Fallback dimensions (common for chatterbox models)
            vocab_size, hidden_size = 32000, 1536
            print(f"Warning: Could not determine dimensions, using defaults: vocab_size={vocab_size}, hidden_size={hidden_size}")
        
        dummy_wte = torch.nn.Embedding(vocab_size, hidden_size)
        t3_state_dict["tfmr.wte.weight"] = dummy_wte.weight.detach().clone()
        print(f"Added dummy tfmr.wte.weight with shape ({vocab_size}, {hidden_size})")
    else:
        print("tfmr.wte.weight already exists in checkpoint")

# Replace the T3 model with the finetuned version
t3_destination = Path(temp_dir) / original_t3_filename
print(f"Saving processed T3 model to {t3_destination}...")
save_file(t3_state_dict, t3_destination)

print(f"✓ Replaced T3 model with finetuned version")

## 5. Create HuggingFace Repository

In [ ]:
# Initialize HuggingFace API
api = HfApi()

# Create repository (will skip if already exists)
try:
    create_repo(
        repo_id=hf_repo_name,
        private=private_repo,
        exist_ok=True
    )
    print(f"✓ Repository '{hf_repo_name}' ready (private={private_repo})")
except Exception as e:
    print(f"Error creating repository: {e}")
    raise

## 6. Upload to HuggingFace

In [ ]:
print(f"Uploading model to {hf_repo_name}...")

try:
    upload_folder(
        folder_path=temp_dir,
        repo_id=hf_repo_name,
        repo_type="model",
        commit_message="Upload finetuned T3 model"
    )
    print(f"\n✓ Successfully uploaded to https://huggingface.co/{hf_repo_name}")
except Exception as e:
    print(f"Error during upload: {e}")
    raise

## 7. Cleanup

In [ ]:
# Clean up temporary directory
print(f"Cleaning up temporary directory: {temp_dir}")
shutil.rmtree(temp_dir)
print("✓ Cleanup complete")